# Numerical computation of modes in antiresonant fibers

Antiresonant fibers (ARFs) guide light in an air core surrounded by a microstructure of thin glass tubes. Unlike step-index fibers that rely on total internal reflection, ARFs rely on the phenomenon of *antiresonance*. The thin glass walls of the capillaries are designed to be "antiresonant" at the desired operating wavelength, which reflects light of that wavelength back into the air core.

One benefit of this geometry is that high-power lasing in an air core entirely evades optical nonlinearities arising from heat or pressure (e.g. TMI and SBS, respectively). One drawback is that antiresonant reflection is not a perfect mechanism of guidance like TIR. This means that the "guided" modes used for communication/lasing are lossy (and their propagation constants are complex), which can cause problems in the kW regime such as melting of the polymer sheath surrounding the fiber.

The `fibermode` module provides the `ARF` class to model these leaky modes using finite elements.

## Initializing ARF instances

The `ARF` class comes with preset geometries for standard fibers from the literature. In particular, `poletti` (6-capillary) and `kolyadin` (8-capillary) are from papers written by people of the same name [citations]. You can also customize the geometric parameters manually by declaring a list.

### 6-capillary fiber

By default, initializing `ARF` without a name creates a model based on the `poletti` 6-capillary design. The design parameters are available upon printing.

In [ ]:
from fibermode.arf import ARF
from ngsolve.webgui import Draw

In [ ]:
arf6 = ARF(name='poletti', refine=0, freecapil=False)
print(arf6)

And like usual we can print using NGSolve webgui:

In [ ]:
Draw(arf6.mesh)

### 8-capillary fiber
The `kolyadin` fiber geometry has 8 capillaries and operates at a longer wavelength.

In [ ]:
arf8 = ARF(name='kolyadin')
print(arf8)
Draw(arf8.mesh)

## Computing modes of ARFs

Since ARF modes are leaky, we use the `leakyvecmodes` method from `modesolver` to solve the Maxwell eigenproblem (or `leakymode` for Helmholtz).

We will search for the fundamental mode of the 6-capillary fiber in the vector case:

In [ ]:
outermaterials = 'air'

center = 5       
radius = 0.2     
alpha = 3
p = 1

betas, zsqrs, E, phi, _ = arf6.leakyvecmodes(
    ctr=center, rad=radius, alpha=alpha,
    nspan=4, npts=4, p=p, niterations=20, nrestarts=0, stop_tol=1e-12, verbose=False
)

Draw(E[0], arf6.mesh)

And also in the scalar case:

In [ ]:
a = ARF(name='poletti', outermaterials='air', freecapil=False)
z, y, yl, beta, P, _ = a.leakymode(p=2,ctr=2.24,rad=0.02,nspan=10,alpha=5,verbose=False)
Draw(y[0],a.mesh)

Above you can observe antiresonance in action. The field amplitude is high in the core and low in the glass boundaries. The leakage loss can be calculated from the imaginary part of the computed eigenvalues.

We can do the same for the 8-capillary fiber:

In [ ]:
arf8 = ARF(name='kolyadin', freecapil=False, refine=0)
Z, y, yl, beta, P, _ = arf8.leakymode(p=3, ctr=2.29, rad=0.02, npts=4, alpha=5)
Draw(y[0], arf8.mesh)